# use pretrained Resnet model to generate latent space, use Conv_autoencoder to get latent space of other sensors

data: only labeled data, umineko 2018 back 1

In [1]:
import torch.optim as optim
from umap import UMAP
import plotly.express as px

import pickle
import numpy as np
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader


from deepview.calculate_results.models.utils import (
    Resnet,
sliding_window,
data_loader_umineko,
load_weights,
MSEloss,
MSEloss_weighted,
Classify_eval_time_series,
Classify_train_time_series_resnet,
majority_value,
# np,
adjust_learning_rate,
# tqdm
Autoencoder3d,
Autoencoder1d,
Autoencoder2d,
Autoencoder3d4
)

In [2]:
def standardization(input_array, mean, std, bias=0.0):
    return ((input_array - mean) / np.maximum(std, 10 ** -5)) + bias

In [3]:
datap = r'D:\code\DeepView\deepview\calculate_results\viz\pamap2\pamap2.npy'
selected_np = np.load(datap)
data_np = selected_np[:,:-1]
print(data_np.shape)
mean = np.mean(data_np, axis=0)
std = np.std(data_np, axis=0)
print("mean:",mean)
print("std:",std)
tmp_b_stand = standardization(data_np, mean, std)

selected_np[:,:-1] = tmp_b_stand
arr = np.unique(selected_np[:,-1])
mapping_dict = {value: idx for idx, value in enumerate(arr)}
selected_np[:,-1] = np.array([mapping_dict[i] for i in selected_np[:,-1]])

device = 'cuda'
len_sw = 300
tmp_b = sliding_window(selected_np, len_sw, len_sw)
# print(tmp_b.shape)

# concatenate list
data_b = np.transpose(tmp_b[:, :, :-1], (0, 2, 1))  # [B, Len, dim-1] -> [B, dim-1, Len]
label_b = tmp_b[:, :, -1]  # [B, Len]

batch_size = 512
train_set_r = data_loader_umineko(data_b, label_b, device=device)
train_loader = DataLoader(train_set_r, batch_size=batch_size,
                          shuffle=False, drop_last=False)

(647624, 3)
mean: [-4.94275048  3.57804587  3.60542344]
std: [6.21694178 6.87116418 3.94584601]


In [5]:
model = Autoencoder3d4(is_reconst=False, is_classify=True)
model = model.to(device)
# 
full_model_path = r'D:\code\DeepView\deepview\calculate_results\viz\AE3d4_reconstruct_epoch500_datalen300_accel.pth'

load_weights(full_model_path, model, my_device=device, is_dist=True, name_start_idx=0)

sensor_type = 'pamap2'

23 Weights loaded


D:\code\DeepView\deepview\calculate_results\models\utils.py:651: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_dict = torch.load(weight_path, map_location=my_devi

In [6]:
criterion = torch.nn.CrossEntropyLoss()
criterion = criterion.to(device)

learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [7]:
from sklearn.metrics import f1_score 
 # training
start_epoch = 0
num_epochs = 500
# lr = 0.0001

for epoch in tqdm(range(start_epoch, num_epochs)):

    learning_rate = adjust_learning_rate(
        learning_rate, optimizer, epoch, p_scheduler='cosine', p_epochs=num_epochs)

    losses,outputs,labels = Classify_train_time_series_resnet(train_loader, model, criterion, optimizer, epoch, scheduler, device)
    
    out = np.concatenate(outputs)
    lab = np.concatenate(labels)
    true_classes = np.argmax(out, axis=1)  # Get the class index from one-hot encoding
    # Compute F1 score using sklearn
    f1 = f1_score(true_classes, lab, average='weighted')
    # print(f"F1 Score: {f1}")
    # break
    # if (epoch % 10 == 0) or (epoch == num_epochs - 1):
    #     print('loss of the ' + str(epoch) + '-th training epoch is :' + losses.__str__())

# print('Saving model at: ' + 'class_ssl_pretrain_epoch%s' % str(epoch) \
#       + '_datalen%s_' % str(len_sw) +sensor_type+ '.pth')
# torch.save(model.state_dict(), 'class_ssl_pretrain_epoch%s' % str(epoch) + \
#            '_datalen%s_' % str(len_sw) +sensor_type+ '.pth')

  0%|          | 0/500 [00:00<?, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:

representation_list, sample_list, pred_list, label_list = \
            Classify_eval_time_series(train_loader, model, device)

# tsne latent representation to shape=(2, len) PCA降维到形状为 (2, len)
repre_concat = np.concatenate(representation_list)
repre_reshape = repre_concat.reshape(repre_concat.shape[0], -1)

sample_concat = np.concatenate(sample_list)
# sample_reshape = sample_concat.reshape(-1, 3)
sample_concat = sample_concat.transpose(0,2,1)
sample_reshape = sample_concat.reshape(-1, sample_concat.shape[-1])

# pred_concat = np.concatenate(pred_list)
# # pred_reshape = pred_concat.reshape(-1, 3)
# pred_concat = pred_concat.transpose(0, 2, 1)
# pred_reshape = pred_concat.reshape(-1, pred_concat.shape[-1])

label_concat = np.concatenate(label_list)
label_concat_vote = majority_value(label_concat)
# label_concat_vote.shape


In [9]:

umap_3d = UMAP(n_components=3)

proj_3d_gyro = umap_3d.fit_transform(repre_reshape)

# set point size 
point_size = np.ones(proj_3d_gyro.shape[0]) * 1
grey_idx = np.where(label_concat_vote==-2)[0]
point_size[grey_idx] = 0.5

fig_3d = px.scatter_3d(
    proj_3d_gyro, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'},
    # color_discrete_map={ '-2.0': ('rgba(239, 239, 240, 1)')},
    color_discrete_map={ '-2.0': 'grey'},
    size=point_size
)

# Update transparency for traces where activity is '-2.0'
fig_3d.for_each_trace(lambda trace: trace.update(marker=dict(opacity=0.5)) if trace.name == '-2.0' else ())

fig_3d.update_traces(marker=dict(line=dict(width=0)))  # remove boundary of point

fig_3d.show()

NameError: name 'repre_reshape' is not defined